<a href="https://colab.research.google.com/github/lightcode01-oss/IYB_INTERNSHIP/blob/main/Day_07_Preprocessing_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DAY 07 — Data Preprocessing & Encoding Pipeline

## YBI Foundation — Python, AI & Data Science Internship

### Objective

The objective of Day 07 is to transform the prepared Student Performance
dataset into a machine-learning-ready format.

### Today's Work

- Load the dataset independently.
- Recreate the project target.
- Remove target leakage.
- Split the data into training and testing sets.
- Identify numerical, nominal, binary and ordinal features.
- Apply appropriate encoding techniques.
- Build a reproducible preprocessing pipeline.
- Transform training and testing data.
- Verify the transformed dataset.
- Check that the preprocessing pipeline is ready for machine learning.

No machine learning model will be trained today.

In [1]:
!pip -q install ucimlrepo

import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
student_performance = fetch_ucirepo(id=320)

X = student_performance.data.features
y_original = student_performance.data.targets

df = X.copy()
df["G3"] = y_original["G3"]

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

Dataset loaded successfully.
Dataset shape: (649, 31)


In [3]:
def performance_category(grade):
    if grade < 10:
        return "Low"
    elif grade < 15:
        return "Medium"
    else:
        return "High"

df["performance_category"] = df["G3"].apply(
    performance_category
)

print(df["performance_category"].value_counts())

performance_category
Medium    418
High      131
Low       100
Name: count, dtype: int64


In [4]:
target = "performance_category"

X = df.drop(columns=[target])
y = df[target]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (649, 31)
Target shape: (649,)


In [5]:
print("G3 present before removal:", "G3" in X.columns)

X_model = X.drop(columns=["G3"])

print("G3 present after removal:", "G3" in X_model.columns)
print("Final feature count:", X_model.shape[1])

G3 present before removal: True
G3 present after removal: False
Final feature count: 30


In [6]:
numerical_features = [
    "age",
    "absences"
]

print("Numerical features:")
print(numerical_features)

Numerical features:
['age', 'absences']


In [7]:
ordinal_features = [
    "Medu",
    "Fedu",
    "traveltime",
    "studytime",
    "failures",
    "famrel",
    "freetime",
    "goout",
    "Dalc",
    "Walc",
    "health"
]

print("Ordinal features:")
print(ordinal_features)

Ordinal features:
['Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health']


In [8]:
binary_features = [
    "school",
    "sex",
    "address",
    "famsize",
    "Pstatus",
    "schoolsup",
    "famsup",
    "paid",
    "activities",
    "nursery",
    "higher",
    "internet",
    "romantic"
]

print("Binary features:")
print(binary_features)

Binary features:
['school', 'sex', 'address', 'famsize', 'Pstatus', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']


In [9]:
nominal_features = [
    "Mjob",
    "Fjob",
    "reason",
    "guardian"
]

print("Nominal features:")
print(nominal_features)

Nominal features:
['Mjob', 'Fjob', 'reason', 'guardian']


In [10]:
all_feature_groups = (
    numerical_features
    + ordinal_features
    + binary_features
    + nominal_features
)

print("Total grouped features:", len(all_feature_groups))
print("Actual model features:", len(X_model.columns))

missing_from_groups = set(X_model.columns) - set(all_feature_groups)
extra_in_groups = set(all_feature_groups) - set(X_model.columns)

print("\nFeatures missing from groups:")
print(missing_from_groups)

print("\nExtra features in groups:")
print(extra_in_groups)

Total grouped features: 30
Actual model features: 30

Features missing from groups:
set()

Extra features in groups:
set()


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training samples: 519
Testing samples: 130

Training target distribution:
performance_category
Medium    334
High      105
Low        80
Name: count, dtype: int64

Testing target distribution:
performance_category
Medium    84
High      26
Low       20
Name: count, dtype: int64


In [12]:
categorical_features = binary_features + nominal_features

categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

print("Categorical encoder created successfully.")

Categorical encoder created successfully.


In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_encoder,
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features + ordinal_features
        )
    ],
    remainder="drop"
)

print("ColumnTransformer created successfully.")

ColumnTransformer created successfully.


In [14]:
preprocessing_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor)
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [15]:
X_train_processed = preprocessing_pipeline.fit_transform(X_train)

print("Training data preprocessing completed.")
print("Processed training shape:", X_train_processed.shape)

Training data preprocessing completed.
Processed training shape: (519, 56)


In [16]:
X_test_processed = preprocessing_pipeline.transform(X_test)

print("Testing data preprocessing completed.")
print("Processed testing shape:", X_test_processed.shape)

Testing data preprocessing completed.
Processed testing shape: (130, 56)


In [17]:
print("Original training shape:")
print(X_train.shape)

print("\nProcessed training shape:")
print(X_train_processed.shape)

print("\nOriginal testing shape:")
print(X_test.shape)

print("\nProcessed testing shape:")
print(X_test_processed.shape)

Original training shape:
(519, 30)

Processed training shape:
(519, 56)

Original testing shape:
(130, 30)

Processed testing shape:
(130, 56)


In [18]:
feature_names = preprocessing_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

print("Number of processed features:", len(feature_names))

print("\nFirst 30 processed feature names:")
print(feature_names[:30])

Number of processed features: 56

First 30 processed feature names:
['categorical__school_GP' 'categorical__school_MS' 'categorical__sex_F'
 'categorical__sex_M' 'categorical__address_R' 'categorical__address_U'
 'categorical__famsize_GT3' 'categorical__famsize_LE3'
 'categorical__Pstatus_A' 'categorical__Pstatus_T'
 'categorical__schoolsup_no' 'categorical__schoolsup_yes'
 'categorical__famsup_no' 'categorical__famsup_yes' 'categorical__paid_no'
 'categorical__paid_yes' 'categorical__activities_no'
 'categorical__activities_yes' 'categorical__nursery_no'
 'categorical__nursery_yes' 'categorical__higher_no'
 'categorical__higher_yes' 'categorical__internet_no'
 'categorical__internet_yes' 'categorical__romantic_no'
 'categorical__romantic_yes' 'categorical__Mjob_at_home'
 'categorical__Mjob_health' 'categorical__Mjob_other'
 'categorical__Mjob_services']


In [19]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print("Processed training DataFrame:")
display(X_train_processed_df.head())

Processed training DataFrame:


,categorical__school_GP,categorical__school_MS,categorical__sex_F,categorical__sex_M,categorical__address_R,categorical__address_U,categorical__famsize_GT3,categorical__famsize_LE3,categorical__Pstatus_A,categorical__Pstatus_T,...,numerical__Fedu,numerical__traveltime,numerical__studytime,numerical__failures,numerical__famrel,numerical__freetime,numerical__goout,numerical__Dalc,numerical__Walc,numerical__health
102,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,...,4.0,1.0,1.0,0.0,5.0,3.0,3.0,1.0,1.0,5.0
346,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,...,4.0,2.0,1.0,0.0,4.0,1.0,1.0,2.0,2.0,5.0
635,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,1.0,1.0,2.0,0.0,4.0,5.0,4.0,2.0,3.0,1.0
628,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,...,2.0,1.0,3.0,0.0,3.0,4.0,3.0,1.0,1.0,3.0
337,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,...,3.0,1.0,3.0,0.0,4.0,2.0,2.0,1.0,2.0,3.0


In [20]:
print("Processed data information:")
print(X_train_processed_df.info())

Processed data information:
<class 'pandas.core.frame.DataFrame'>
Index: 519 entries, 102 to 477
Data columns (total 56 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   categorical__school_GP          519 non-null    float64
 1   categorical__school_MS          519 non-null    float64
 2   categorical__sex_F              519 non-null    float64
 3   categorical__sex_M              519 non-null    float64
 4   categorical__address_R          519 non-null    float64
 5   categorical__address_U          519 non-null    float64
 6   categorical__famsize_GT3        519 non-null    float64
 7   categorical__famsize_LE3        519 non-null    float64
 8   categorical__Pstatus_A          519 non-null    float64
 9   categorical__Pstatus_T          519 non-null    float64
 10  categorical__schoolsup_no       519 non-null    float64
 11  categorical__schoolsup_yes      519 non-null    float64
 12  categorical

In [21]:
print("\nSample processed rows:")
display(X_train_processed_df.head(10))


Sample processed rows:


,categorical__school_GP,categorical__school_MS,categorical__sex_F,categorical__sex_M,categorical__address_R,categorical__address_U,categorical__famsize_GT3,categorical__famsize_LE3,categorical__Pstatus_A,categorical__Pstatus_T,...,numerical__Fedu,numerical__traveltime,numerical__studytime,numerical__failures,numerical__famrel,numerical__freetime,numerical__goout,numerical__Dalc,numerical__Walc,numerical__health
102,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,...,4.0,1.0,1.0,0.0,5.0,3.0,3.0,1.0,1.0,5.0
346,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,...,4.0,2.0,1.0,0.0,4.0,1.0,1.0,2.0,2.0,5.0
635,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,1.0,1.0,2.0,0.0,4.0,5.0,4.0,2.0,3.0,1.0
628,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,...,2.0,1.0,3.0,0.0,3.0,4.0,3.0,1.0,1.0,3.0
337,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,...,3.0,1.0,3.0,0.0,4.0,2.0,2.0,1.0,2.0,3.0
543,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,...,4.0,3.0,1.0,3.0,3.0,3.0,3.0,1.0,3.0,5.0
535,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,...,2.0,3.0,2.0,0.0,5.0,5.0,5.0,1.0,3.0,5.0
427,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,3.0,2.0,2.0,0.0,4.0,5.0,2.0,1.0,2.0,5.0
31,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,...,4.0,2.0,2.0,0.0,4.0,3.0,1.0,1.0,1.0,5.0
152,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,...,2.0,2.0,1.0,0.0,2.0,3.0,3.0,2.0,2.0,2.0


In [22]:
print(
    "Missing values in processed training data:",
    X_train_processed_df.isnull().sum().sum()
)

print(
    "Missing values in processed testing data:",
    X_test_processed_df.isnull().sum().sum()
)

Missing values in processed training data: 0
Missing values in processed testing data: 0


In [23]:
print("Is G3 present in processed features?")
print(any("G3" in feature for feature in feature_names))

print("\nIs performance_category present?")
print(any("performance_category" in feature for feature in feature_names))

Is G3 present in processed features?
False

Is performance_category present?
False


In [24]:
preprocessing_summary = pd.DataFrame({
    "Dataset": [
        "Original Training",
        "Processed Training",
        "Original Testing",
        "Processed Testing"
    ],
    "Rows": [
        X_train.shape[0],
        X_train_processed_df.shape[0],
        X_test.shape[0],
        X_test_processed_df.shape[0]
    ],
    "Columns": [
        X_train.shape[1],
        X_train_processed_df.shape[1],
        X_test.shape[1],
        X_test_processed_df.shape[1]
    ]
})

preprocessing_summary

,Dataset,Rows,Columns
0,Original Training,519,30
1,Processed Training,519,56
2,Original Testing,130,30
3,Processed Testing,130,56


In [25]:
X_test_processed_again = preprocessing_pipeline.transform(X_test)

print(
    "Repeated transformation identical:",
    np.array_equal(
        X_test_processed,
        X_test_processed_again
    )
)

Repeated transformation identical: True


In [26]:
print("=" * 60)
print("DAY 07 PREPROCESSING CHECK")
print("=" * 60)

print("Original samples:", len(X_model))
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nOriginal feature count:", X_model.shape[1])
print("Processed feature count:", X_train_processed_df.shape[1])

print("\nMissing values after preprocessing:")
print(X_train_processed_df.isnull().sum().sum())

print("\nTarget classes:")
print(y.unique())

print("\nG3 leakage removed:", "G3" not in X_model.columns)

print("\nPreprocessing pipeline ready:", True)

DAY 07 PREPROCESSING CHECK
Original samples: 649
Training samples: 519
Testing samples: 130

Original feature count: 30
Processed feature count: 56

Missing values after preprocessing:
0

Target classes:
['Medium' 'High' 'Low']

G3 leakage removed: True

Preprocessing pipeline ready: True


# Day 07 — Preprocessing Findings

## Data Splitting

The dataset was divided into training and testing subsets using an 80:20
split with stratification. Stratification was used to preserve the relative
distribution of the three performance categories.

## Categorical Encoding

Categorical variables were converted into numerical representations using
One-Hot Encoding.

The encoder was configured with `handle_unknown="ignore"` so that unseen
categories during future prediction do not cause preprocessing errors.

## Numerical and Ordinal Features

Numerical features such as `age` and `absences` were retained as numerical
values.

Ordinal or rating-style features such as `studytime`, `failures`, `Medu`,
`Fedu`, `famrel`, `freetime`, `goout`, `Dalc`, `Walc` and `health` were
retained in their existing ordered numerical representation.

## Preprocessing Pipeline

A Scikit-learn `ColumnTransformer` and `Pipeline` were created to make the
preprocessing process reproducible.

The pipeline is fitted only on the training data and then used to transform
the testing data.

## Target Leakage

The final grade `G3` was excluded from the model features because the target
`performance_category` is directly derived from `G3`.

This prevents the model from receiving direct information about the target.

## Verification

The processed training and testing datasets were checked for missing values,
feature dimensions and target leakage.

Repeated transformation of the same test data produced identical results,
confirming consistent preprocessing behavior.

## Conclusion

Day 07 successfully converted the prepared dataset into a machine-learning-
ready format.

The preprocessing pipeline can now be reused during model training and later
during prediction on new student records.

# DAY 07 DEVELOPMENT LOG

## Work Completed

- Loaded the Student Performance dataset independently.
- Recreated the project performance categories.
- Separated input features and target.
- Removed `G3` from the model feature set.
- Defined numerical features.
- Defined ordinal features.
- Defined binary categorical features.
- Defined nominal categorical features.
- Verified that every model feature belongs to a preprocessing group.
- Split the dataset into training and testing sets.
- Used stratified train-test splitting.
- Created a One-Hot Encoder.
- Created a Scikit-learn ColumnTransformer.
- Created a reproducible preprocessing Pipeline.
- Fitted preprocessing only on training data.
- Transformed training and testing data.
- Generated processed feature names.
- Converted processed arrays into DataFrames for inspection.
- Checked for missing values after preprocessing.
- Verified that `G3` and the target were not included in processed features.
- Verified preprocessing reproducibility.

## Key Learning

Different feature types require different preprocessing strategies.

One-hot encoding is useful for categorical variables without numerical
meaning, while ordered numerical features can retain their existing
representation when appropriate.

Using a preprocessing pipeline helps ensure that the same transformations
are applied consistently during training and future prediction.

## Important Project Decision

The preprocessing pipeline will be reused during future model training and
prediction instead of manually transforming individual columns.

## Next Step

Day 08 will focus on establishing the baseline machine learning setup and
preparing the first classification model using the preprocessing pipeline.